# CSE 151B — SFT evaluation harness

Evaluates **LoRA checkpoints** on frozen eval slices (same as `notebooks/sft_train.ipynb` §8): **holdout** (`holdout_20p.jsonl`, 225 rows), **geometry** (`geometry_dev.jsonl`), **prob_stats** (`prob_stats_dev.jsonl`). Same vLLM + `Judger` path as `notebooks/dev.ipynb` — **A100 bf16**, **`multi_blank`** prompts, 16k generation — with SFT monitoring metrics. Holdout run also reports watch subsets from `data/eval/watch_manifest.json` (Q4 long, multi-blank ≥3, geometry/sequences overlap).

1. (**Colab A100**) Copy `data/eval/` holdout + dev slices + watch files from Drive.
2. Load base model + optional LoRA adapter (bf16 vLLM).
3. Loop slices: generate → score → extended summary → save.
4. Write `results/sft_eval/<run_name>/eval_<slice>_<step>.json` on Drive; gate on holdout overall ≥ 64.44%, MCQ ≥ 77%.

Set `LORA_PATH = ""` for baseline-only eval. Set `EVAL_SLICE_FILTER` to run a single slice (e.g. `"holdout"`).


## 1. Environment

**Colab A100:** run the `%pip` cell below, restart runtime, continue. **Local:** use your venv with `vllm`, `transformers`, etc. — **no `bitsandbytes`** (bf16 native load; see [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md)).


In [1]:
# Colab: skip when working locally with an existing venv.
%pip install -q sympy numpy tqdm "antlr4-python3-runtime==4.11.1"
%pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu129 --upgrade
%pip install -q "https://github.com/vllm-project/vllm/releases/download/v0.20.0/vllm-0.20.0+cu129-cp38-abi3-manylinux_2_31_x86_64.whl" --extra-index-url https://download.pytorch.org/whl/cu129
%pip install -q transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 15.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.1 which is incompatible.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.8/647.8 MB 86.8 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.2/581.2 MB 81.7 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 220.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.9/200.9 MB 129.2 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 243.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 211.1 MB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.3/68.3 MB 178.2 MB/s eta 0:00:00a 0:00:01
     ━━━━

## 2. Imports & configuration

SFT keys: `LORA_PATH`, `SFT_RUN_NAME`, `EVAL_STEP`, `EVAL_SLICES`, `EVAL_SLICE_FILTER`, `PROMPT_VARIANT`, `MAX_TOKENS`, `BASELINE_MEAN_RESPONSE_LEN`. Slice list matches `notebooks/sft_train.ipynb` §8 — never resample here. Defaults match `notebooks/dev.ipynb` (`multi_blank`, 16k).


In [1]:
import json
import os
import random
from pathlib import Path


def _repo_root() -> Path:
    here = Path.cwd().resolve()
    if (here / "data").is_dir():
        return here
    if here.name == "notebooks" and (here.parent / "data").is_dir():
        return here.parent
    up = here.parent
    if (up / "data").is_dir():
        return up
    return here


REPO_ROOT = _repo_root()

# --- eval slices (match sft_train.ipynb §8) ---
GATE_HOLDOUT_OVERALL = 0.6444
GATE_HOLDOUT_MCQ = 0.77
PUB002_GEOMETRY_ACC = 53.38
PUB002_PROB_STATS_ACC = 49.8
EVAL_SLICES = [
    ("holdout", REPO_ROOT / "data/eval/holdout_20p.jsonl"),
    ("geometry", REPO_ROOT / "data/eval/geometry_dev.jsonl"),
    ("prob_stats", REPO_ROOT / "data/eval/prob_stats_dev.jsonl"),
]
# Set to a slice name to run one slice only; None = all EVAL_SLICES
EVAL_SLICE_FILTER = None

LORA_PATH     = "checkpoints/openmath_sft007_5k/final_adapter"
SFT_RUN_NAME  = "openmath_sft007_5k"
EVAL_STEP     = 0
BASELINE_MEAN_RESPONSE_LEN = 14000

PROMPT_VARIANT = "multi_blank"  # match dev.ipynb default (roadmap §1.3)
MODEL_ID    = "Qwen/Qwen3-4B-Thinking-2507"
GPU_ID      = "0"
MAX_TOKENS  = 16384
SEED        = 42

print(f"REPO_ROOT={REPO_ROOT}")
print(f"LORA_PATH={LORA_PATH or '(none — base model)'}")
print(f"EVAL_SLICES={[s for s, _ in EVAL_SLICES]}  filter={EVAL_SLICE_FILTER!r}")
print(f"PROMPT_VARIANT={PROMPT_VARIANT!r}  MAX_TOKENS={MAX_TOKENS}")

os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "0"

import glob
import site

def _prepend_nvidia_libs_to_ld_path() -> None:
    roots = list(site.getsitepackages())
    user_site = getattr(site, "getusersitepackages", lambda: None)()
    if user_site:
        roots.append(user_site)
    libdirs: list[str] = []
    seen: set[str] = set()
    for root in roots:
        for d in glob.glob(os.path.join(root, "nvidia", "*", "lib")):
            if os.path.isdir(d) and d not in seen:
                seen.add(d)
                libdirs.append(d)
    if libdirs:
        sep = os.pathsep
        os.environ["LD_LIBRARY_PATH"] = sep.join(libdirs + [os.environ.get("LD_LIBRARY_PATH", "")]).strip(sep)


_prepend_nvidia_libs_to_ld_path()

import contextlib
import io
import re
import sys
from typing import Optional

@contextlib.contextmanager
def _jupyter_stdout_for_vllm():
    try:
        sys.stdout.fileno()
    except (io.UnsupportedOperation, AttributeError, OSError):
        saved_out, saved_err = sys.stdout, sys.stderr
        dup1, dup2 = os.dup(1), os.dup(2)
        try:
            sys.stdout = os.fdopen(dup1, "w", buffering=1)
            sys.stderr = os.fdopen(dup2, "w", buffering=1)
            yield
        finally:
            sys.stdout.close()
            sys.stderr.close()
            sys.stdout, sys.stderr = saved_out, saved_err
    else:
        yield

from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from tqdm import tqdm

REPO_ROOT=/content
LORA_PATH=checkpoints/openmath_sft007_5k/final_adapter
EVAL_SLICES=['holdout', 'geometry', 'prob_stats']  filter=None
PROMPT_VARIANT='multi_blank'  MAX_TOKENS=16384


## 3. Colab: copy eval slices + watch sets from Drive

Canonical snapshots under `CSE151B/data/eval/` on Drive (holdout, geometry, prob_stats, watch manifest).


In [2]:
import shutil
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    print("Skip: not Google Colab — use repo `data/eval/`.")
    DRIVE_PROJECT_ROOT = None
else:
    drive.mount("/content/drive")
    DRIVE_PROJECT_ROOT = Path("/content/drive/MyDrive/CSE151B")
    DRIVE_EVAL = DRIVE_PROJECT_ROOT / "data/eval"
    DRIVE_HOLDOUT = DRIVE_EVAL / "holdout_20p.jsonl"
    if not DRIVE_HOLDOUT.is_file():
        raise FileNotFoundError(
            f"Missing on Drive: {DRIVE_HOLDOUT}\n"
            "Upload frozen eval holdout to Drive (data/eval/holdout.jsonl)."
        )
    eval_dir = REPO_ROOT / "data/eval"
    eval_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(DRIVE_HOLDOUT, eval_dir / "holdout_20p.jsonl")
    print(f"Copied holdout.jsonl to {(eval_dir / 'holdout_20p.jsonl').resolve()}")

    for name in (
        "watch_manifest.json",
        "geometry_dev.jsonl",
        "prob_stats_dev.jsonl",
        "sequences_dev.jsonl",
        "watch_q4_long.jsonl",
        "watch_multi_blank_ge3.jsonl",
    ):
        src = DRIVE_EVAL / name
        if src.is_file():
            shutil.copy2(src, eval_dir / name)
            print(f"Copied {name}")

    # Drive manifest may predate geometry watch — backfill from geometry_dev.jsonl
    _manifest = eval_dir / "watch_manifest.json"
    _geo = eval_dir / "geometry_dev.jsonl"
    if _manifest.is_file() and _geo.is_file():
        import json as _json

        _m = _json.loads(_manifest.read_text())
        if "geometry" not in _m.get("watch", {}):
            _geo_ids = [_json.loads(line)["id"] for line in _geo.open()]
            _m.setdefault("watch", {})["geometry"] = {
                "name": "geometry",
                "n": len(_geo_ids),
                "ids": _geo_ids,
            }
            _manifest.write_text(_json.dumps(_m, indent=2) + "\n")
            print(f"Patched watch_manifest.json with geometry ({len(_geo_ids)} ids)")

    _seq = eval_dir / "sequences_dev.jsonl"
    if _manifest.is_file() and _seq.is_file():
        _m = _json.loads(_manifest.read_text())
        if "sequences" not in _m.get("watch", {}):
            _seq_ids = [_json.loads(line)["id"] for line in _seq.open()]
            _m.setdefault("watch", {})["sequences"] = {
                "name": "sequences",
                "n": len(_seq_ids),
                "ids": _seq_ids,
            }
            _manifest.write_text(_json.dumps(_m, indent=2) + "\n")
            print(f"Patched watch_manifest.json with sequences ({len(_seq_ids)} ids)")

    if LORA_PATH and not Path(LORA_PATH).is_absolute():
        _lora = DRIVE_PROJECT_ROOT / LORA_PATH
        if _lora.is_dir():
            LORA_PATH = str(_lora)
            print(f"Resolved LORA_PATH={LORA_PATH}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied holdout.jsonl to /content/data/eval/holdout_20p.jsonl
Copied watch_manifest.json
Copied geometry_dev.jsonl
Copied prob_stats_dev.jsonl
Copied sequences_dev.jsonl
Copied watch_q4_long.jsonl
Copied watch_multi_blank_ge3.jsonl
Resolved LORA_PATH=/content/drive/MyDrive/CSE151B/checkpoints/openmath_sft007_5k/final_adapter


## 4. Load dataset

Frozen `data/eval/holdout.jsonl`. Rebuild via `scripts/build_eval_holdout.py` (not in this notebook).


In [3]:
def n_ans_placeholders(question: str) -> int:
    return question.count("[ANS]")


_preview_path = REPO_ROOT / "data/eval/holdout_20p.jsonl"
data = [json.loads(line) for line in open(_preview_path)]

n_mcq  = sum(bool(d.get("options")) for d in data)
n_free = sum(not d.get("options") for d in data)
print(f"Preview {len(data)} holdout questions  ({n_mcq} MCQ, {n_free} free-form) from {_preview_path}")

mcq_sample  = next((d for d in data if d.get("options")), None)
free_sample = next((d for d in data if not d.get("options")), None)
multi_sample = next(
    (d for d in data if not d.get("options") and n_ans_placeholders(d["question"]) > 1),
    free_sample,
)

if mcq_sample:
    print("\n── MCQ sample ──")
    print(json.dumps(mcq_sample, indent=2)[:1200], "...\n" if len(json.dumps(mcq_sample)) > 1200 else "\n")
if free_sample:
    print("── Free-form sample ──")
    print(json.dumps(free_sample, indent=2)[:1200], "...\n" if len(json.dumps(free_sample)) > 1200 else "\n")
if multi_sample and multi_sample is not free_sample:
    print(f"── Multi-blank sample ({n_ans_placeholders(multi_sample['question'])} blanks) ──")
    print(json.dumps(multi_sample, indent=2)[:1200], "...\n" if len(json.dumps(multi_sample)) > 1200 else "\n")

Preview 225 holdout questions  (75 MCQ, 150 free-form) from /content/data/eval/holdout_20p.jsonl

── MCQ sample ──
{
  "question": "An ordinary deck of cards containing 26 red cards and 26 black cards is shuffled and dealt out one card at a time without replacement. Let $X_i$ be the color of the $i$th card. Compute $H(X_1,X_2,\\ldots,X_{52})$ in bits.",
  "options": [
    "52",
    "48.8",
    "49.9",
    "50.0",
    "51.5",
    "46.5",
    "50.2",
    "47.3",
    "45.6",
    "53.2"
  ],
  "answer": "B",
  "id": 33
} 

── Free-form sample ──
{
  "question": "Reduce the fraction ${\\frac{25}{40}}$. [ANS]",
  "answer": [
    "5/8"
  ],
  "id": 3
} 

── Multi-blank sample (2 blanks) ──
{
  "question": "Let $p$ be the price of an item and $q$ be the number of items sold at that price, where $q=f(p)$. What do the following quantities mean in terms of prices and quantities sold? 1. $f(45)$ is the [ANS] A. price for which 45 items are sold  B. rate at which the price is changing when 45 items

## 5. Prompt construction

Same as `notebooks/dev.ipynb` §6 — controlled by `PROMPT_VARIANT` (default `"multi_blank"`): separate `\boxed{}` per `[ANS]` blank, comma-separated in blank order.


In [4]:
_MATH_BASELINE = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "Put your final answer inside \\boxed{}. "
    "If the problem has multiple sub-answers, separate them by commas inside a single \\boxed{}, "
    "e.g. \\boxed{3, 7}."
)

_MCQ_BASELINE = (
    "You are an expert mathematician. "
    "Read the problem and the answer choices below, then select the single best answer. "
    "Output ONLY the letter of your chosen option inside \\boxed{}, e.g. \\boxed{C}."
)

_MATH_MULTI_BLANK = (
    "You are an expert mathematician. Solve the problem step-by-step. "
    "For problems with multiple [ANS] placeholders, put each sub-answer in its own "
    "\\boxed{}, separated by commas, in the order the blanks appear "
    "(e.g. \\boxed{3}, \\boxed{7}). Do not use labels like 'Answer 1:' between boxes. "
    "For single-answer problems, use one \\boxed{}."
)

_MCQ_MULTI_BLANK = _MCQ_BASELINE

_PROMPTS = {
    "baseline": (_MATH_BASELINE, _MCQ_BASELINE),
    "multi_blank": (_MATH_MULTI_BLANK, _MCQ_MULTI_BLANK),
}
assert PROMPT_VARIANT in _PROMPTS, f"Unknown PROMPT_VARIANT {PROMPT_VARIANT!r}; choose from {list(_PROMPTS)}"
SYSTEM_PROMPT_MATH, SYSTEM_PROMPT_MCQ = _PROMPTS[PROMPT_VARIANT]


def build_prompt(question: str, options: Optional[list]) -> tuple[str, str]:
    if options:
        labels = [chr(65 + i) for i in range(len(options))]
        opts_text = "\n".join(f"{lbl}. {opt.strip()}" for lbl, opt in zip(labels, options))
        return SYSTEM_PROMPT_MCQ, f"{question}\n\nOptions:\n{opts_text}"
    n_blanks = n_ans_placeholders(question)
    user = question
    if PROMPT_VARIANT == "multi_blank" and n_blanks > 1:
        example = ", ".join("\\boxed{...}" for _ in range(n_blanks))
        user = (
            f"{question}\n\n"
            f"The problem has {n_blanks} [ANS] blanks. "
            f"After your reasoning, give {n_blanks} comma-separated \\boxed{{}} values "
            f"in order: {example}"
        )
    return SYSTEM_PROMPT_MATH, user


print(f"Active variant: {PROMPT_VARIANT!r}")

_demo_items = []
if mcq_sample:
    _demo_items.append(("MCQ", mcq_sample))
if free_sample:
    _demo_items.append(("Free-form", free_sample))
if multi_sample and multi_sample not in (mcq_sample, free_sample):
    _demo_items.append((f"Multi-blank ({n_ans_placeholders(multi_sample['question'])} blanks)", multi_sample))
for label, item in _demo_items:
    sys_p, usr_p = build_prompt(item["question"], item.get("options"))
    print(f"── {label} user prompt (first 300 chars) ──")
    print(usr_p[:300], "...\n")

Active variant: 'multi_blank'
── MCQ user prompt (first 300 chars) ──
An ordinary deck of cards containing 26 red cards and 26 black cards is shuffled and dealt out one card at a time without replacement. Let $X_i$ be the color of the $i$th card. Compute $H(X_1,X_2,\ldots,X_{52})$ in bits.

Options:
A. 52
B. 48.8
C. 49.9
D. 50.0
E. 51.5
F. 46.5
G. 50.2
H. 47.3
I. 45.6 ...

── Free-form user prompt (first 300 chars) ──
Reduce the fraction ${\frac{25}{40}}$. [ANS] ...

── Multi-blank (2 blanks) user prompt (first 300 chars) ──
Let $p$ be the price of an item and $q$ be the number of items sold at that price, where $q=f(p)$. What do the following quantities mean in terms of prices and quantities sold? 1. $f(45)$ is the [ANS] A. price for which 45 items are sold  B. rate at which the price is changing when 45 items are sold ...



## 6. Load model with vLLM (A100 profile)

Matches `notebooks/dev.ipynb` §7 — bf16, `max_model_len=17500`, prefix caching, chunked prefill. LoRA when `LORA_PATH` is set. See [`docs/infra/vllm-inference-config.md`](../docs/infra/vllm-inference-config.md).


In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

USE_LORA = bool(LORA_PATH) and Path(LORA_PATH).is_dir()
if LORA_PATH and not USE_LORA:
    raise FileNotFoundError(f"LORA_PATH set but not found: {LORA_PATH}")

_llm_kwargs = dict(
    model=MODEL_ID,
    dtype="bfloat16",
    enable_prefix_caching=True,
    gpu_memory_utilization=0.92,
    max_model_len=17500,
    trust_remote_code=True,
    max_num_seqs=128,
    max_num_batched_tokens=32768,
    enable_chunked_prefill=True,
)
if USE_LORA:
    _llm_kwargs.update(enable_lora=True, max_loras=1, max_lora_rank=64)

with _jupyter_stdout_for_vllm():
    llm = LLM(**_llm_kwargs)

lora_request = None
if USE_LORA:
    from vllm.lora.request import LoRARequest
    lora_request = LoRARequest("sft_adapter", 1, LORA_PATH)
    print(f"LoRA adapter: {LORA_PATH}")
else:
    print("No LoRA — evaluating base model.")

default_sampling_params = SamplingParams(
    max_tokens=MAX_TOKENS,
    temperature=0.6,
    top_p=0.95,
    top_k=20,
    min_p=0.0,
    presence_penalty=0.0,
    repetition_penalty=1.0,
    seed=SEED,
)

print("Model loaded.")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


INFO 05-31 08:42:34 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': 'bfloat16', 'max_model_len': 17500, 'enable_prefix_caching': True, 'max_num_batched_tokens': 32768, 'max_num_seqs': 128, 'disable_log_stats': True, 'enable_lora': True, 'max_lora_rank': 64, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen3-4B-Thinking-2507'}


WARNING 05-31 08:42:34 [arg_utils.py:1467] The global random seed is set to 0. Since VLLM_ENABLE_V1_MULTIPROCESSING is set to False, this may affect the random state of the Python process that launched vLLM.
INFO 05-31 08:42:35 [model.py:555] Resolved architecture: Qwen3ForCausalLM
INFO 05-31 08:42:35 [model.py:1680] Using max model len 17500
INFO 05-31 08:42:35 [scheduler.py:239] Chunked prefill is enabled with max_num_batched_tokens=32768.
INFO 05-31 08:42:35 [vllm.py:840] Asynchronous scheduling is enabled.
INFO 05-31 08:42:35 [kernel.py:205] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'])
INFO 05-31 08:42:39 [core.py:109] Initializing a V1 LLM engine (v0.20.0) with config: model='Qwen/Qwen3-4B-Thinking-2507', speculative_config=None, tokenizer='Qwen/Qwen3-4B-Thinking-2507', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=17500, download_dir

Loading safetensors checkpoint shards:   0% Completed | 0/3 [00:00<?, ?it/s]


INFO 05-31 08:42:46 [default_loader.py:384] Loading weights took 2.61 seconds
INFO 05-31 08:42:46 [punica_selector.py:20] Using PunicaWrapperGPU.
INFO 05-31 08:42:47 [gpu_model_runner.py:4879] Model loading took 7.92 GiB memory and 5.175847 seconds
INFO 05-31 08:42:55 [backends.py:1069] Using cache directory: /root/.cache/vllm/torch_compile_cache/0aa66dbc1d/rank_0_0/backbone for vLLM's torch.compile
INFO 05-31 08:42:55 [backends.py:1128] Dynamo bytecode transform time: 7.20 s
INFO 05-31 08:43:00 [backends.py:290] Directly load the compiled graph(s) for compile range (1, 32768) from the cache, took 3.581 s
INFO 05-31 08:43:00 [decorators.py:305] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/9e74011c72525b9b022a552a66262ec92b40ade91dfc30e894915c9035f7b955/rank_0_0/model
INFO 05-31 08:43:00 [monitor.py:53] torch.compile took 11.55 s in total
INFO 05-31 08:43:00 [monitor.py:81] Initial profiling/warmup run took 0.20 s
INFO 05-31 08:43:11 [g

## 7. Generate responses (all eval slices)

Loops `EVAL_SLICES` (same as `sft_train.ipynb` §8). Checkpoint per slice: `results/sft_eval/<run_name>/eval_<slice>_<step>.responses.jsonl` — delete to regenerate.


In [6]:
def _active_slices():
    for slice_name, data_path in EVAL_SLICES:
        if EVAL_SLICE_FILTER and slice_name != EVAL_SLICE_FILTER:
            continue
        yield slice_name, Path(data_path)


try:
    _results_dir = DRIVE_PROJECT_ROOT / "results"
except NameError:
    _results_dir = Path(REPO_ROOT) / "results"

_results_dir.mkdir(parents=True, exist_ok=True)
_eval_root = _results_dir / "sft_eval" / SFT_RUN_NAME
_eval_root.mkdir(parents=True, exist_ok=True)

slice_data: dict[str, list] = {}
slice_responses: dict[str, list[str]] = {}

for slice_name, data_path in _active_slices():
    if not data_path.is_file():
        raise FileNotFoundError(data_path)
    data = [json.loads(line) for line in open(data_path)]
    slice_data[slice_name] = data

    response_checkpoint = _eval_root / f"eval_{slice_name}_{EVAL_STEP}.responses.jsonl"
    response_checkpoint.parent.mkdir(parents=True, exist_ok=True)
    print(f"\n=== eval slice={slice_name} n={len(data)} checkpoint={response_checkpoint.name} ===")

    completed: dict = {}
    if response_checkpoint.exists():
        with open(response_checkpoint) as f:
            for line in f:
                r = json.loads(line)
                completed[r["id"]] = r["response"]
        print(f"Resumed from checkpoint: {len(completed)} responses already generated")

    remaining = [d for d in data if d.get("id") not in completed]
    print(f"Generating {len(remaining)} remaining / {len(data)} total...")

    CHUNK_SIZE = len(data)
    for chunk_start in range(0, len(remaining), CHUNK_SIZE):
        chunk = remaining[chunk_start : chunk_start + CHUNK_SIZE]

        prompts = []
        for item in chunk:
            system, user = build_prompt(item["question"], item.get("options"))
            prompt_text = tokenizer.apply_chat_template(
                [{"role": "system", "content": system},
                 {"role": "user",   "content": user}],
                tokenize=False,
                add_generation_prompt=True,
            )
            prompts.append(prompt_text)

        chunk_params = [default_sampling_params] * len(prompts)

        with _jupyter_stdout_for_vllm():
            _gen_kwargs = dict(prompts=prompts, sampling_params=chunk_params)
            if lora_request is not None:
                _gen_kwargs["lora_request"] = lora_request
            outputs = llm.generate(**_gen_kwargs)

        with open(response_checkpoint, "a") as f:
            for item, out in zip(chunk, outputs):
                response = out.outputs[0].text.strip()
                completed[item["id"]] = response
                f.write(json.dumps({"id": item["id"], "response": response}) + "\n")

        done = len(completed)
        print(f"  {done}/{len(data)}  ({done / len(data) * 100:.1f}%)")

    slice_responses[slice_name] = [completed[d["id"]] for d in data]
    print(f"Done slice={slice_name}: {len(slice_responses[slice_name])} responses ready.")

print(f"\nGenerated slices: {list(slice_responses.keys())}")



=== eval slice=holdout n=225 checkpoint=eval_holdout_0.responses.jsonl ===


Rendering prompts:   0%|          | 0/205 [00:00<?, ?it/s]

Resumed from checkpoint: 225 responses already generated
Generating 0 remaining / 225 total...
Done slice=holdout: 225 responses ready.

=== eval slice=geometry n=108 checkpoint=eval_geometry_0.responses.jsonl ===
Resumed from checkpoint: 108 responses already generated
Generating 0 remaining / 108 total...
Done slice=geometry: 108 responses ready.

=== eval slice=prob_stats n=205 checkpoint=eval_prob_stats_0.responses.jsonl ===
Generating 205 remaining / 205 total...
WARNING 05-31 08:43:41 [input_processor.py:149] vLLM has deprecated support for supporting different tokenizers for different LoRAs. By default, vLLM uses base model's tokenizer. If you are using a LoRA with its own tokenizer, consider specifying `--tokenizer [lora_path]` to use the LoRA tokenizer.


Processed prompts:   0%|          | 0/205 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

  205/205  (100.0%)
Done slice=prob_stats: 205 responses ready.

Generated slices: ['holdout', 'geometry', 'prob_stats']


## 8. Score responses (same as starter)


In [7]:
_judger_override = os.environ.get("CSE151B_JUDGER_DIR", "").strip()
try:
    _drive_root = str(DRIVE_PROJECT_ROOT)
except NameError:
    _drive_root = ""
JUDGER_ROOT = _judger_override or _drive_root or str(REPO_ROOT)

print(f"JUDGER_ROOT={JUDGER_ROOT}")
sys.path.insert(0, os.path.abspath(JUDGER_ROOT))
from judger import Judger
judger = Judger(strict_extract=False)

JUDGER_ROOT=/content/drive/MyDrive/CSE151B


In [8]:
def extract_letter(text: str) -> str:
    m = re.search(r"\\boxed\{([A-Za-z])\}", text)
    if m:
        return m.group(1).upper()
    matches = re.findall(r"\b([A-Z])\b", text.upper())
    return matches[-1] if matches else ""


def score_mcq(response: str, gold_letter: str) -> bool:
    return extract_letter(response) == gold_letter.strip().upper()


slice_results: dict[str, list] = {}

for slice_name, data in slice_data.items():
    responses = slice_responses[slice_name]
    results = []
    for item, response in tqdm(zip(data, responses), total=len(data), desc=f"Scoring {slice_name}"):
        is_mcq = bool(item.get("options"))
        gold   = item["answer"]

        if is_mcq:
            correct = score_mcq(response, str(gold))
        else:
            gold_list = gold if isinstance(gold, list) else [gold]
            try:
                correct = judger.auto_judge(
                    pred=response,
                    gold=gold_list,
                    options=[[]] * len(gold_list),
                )
            except Exception:
                correct = False

        results.append({
            "id":       item.get("id"),
            "is_mcq":   is_mcq,
            "gold":     gold,
            "response": response,
            "correct":  correct,
        })

    slice_results[slice_name] = results
    print(f"Scoring complete for {slice_name}: {len(results)} results.")

Scoring holdout: 100%|██████████| 225/225 [00:19<00:00, 11.52it/s]


Scoring complete for holdout: 225 results.


Scoring geometry: 100%|██████████| 108/108 [00:09<00:00, 11.34it/s]


Scoring complete for geometry: 108 results.


Scoring prob_stats: 100%|██████████| 205/205 [00:23<00:00,  8.87it/s]

Scoring complete for prob_stats: 205 results.


## 9. Extended summary + holdout gate

Per-slice metrics (extended breakdown on holdout). Gate matches `sft_train.ipynb` §8: holdout overall ≥ 64.44%, MCQ ≥ 77%.


In [9]:
def mcq_has_boxed_letter(response: str, n_options: int) -> bool:
    last = chr(64 + n_options)
    return bool(re.search(rf"\\boxed\{{[A-{last}]\}}", response, re.IGNORECASE))


def acc(subset):
    return sum(r["correct"] for r in subset) / len(subset) * 100 if subset else 0.0


_watch_manifest_path = REPO_ROOT / "data" / "eval" / "watch_manifest.json"
with open(_watch_manifest_path) as f:
    _watch_manifest = json.load(f)


def _backfill_watch_from_dev(watch_name: str, dev_path: Path) -> None:
    if watch_name in _watch_manifest.get("watch", {}) or not dev_path.is_file():
        return
    ids = [json.loads(line)["id"] for line in dev_path.open()]
    _watch_manifest.setdefault("watch", {})[watch_name] = {"ids": ids, "n": len(ids)}
    print(f"watch_manifest missing {watch_name}; backfilled {len(ids)} ids from {dev_path.name}")


_backfill_watch_from_dev("geometry", REPO_ROOT / "data" / "eval" / "geometry_dev.jsonl")
_backfill_watch_from_dev("sequences", REPO_ROOT / "data" / "eval" / "sequences_dev.jsonl")


def _watch_ids(watch_name):
    watch = _watch_manifest.get("watch", {})
    if watch_name in watch:
        return set(watch[watch_name]["ids"])
    raise KeyError(
        f"Unknown watch set {watch_name!r}; rebuild data/eval/watch_manifest.json "
        f"(have: {sorted(watch.keys())})"
    )


def _watch_accuracy(results, watch_name):
    ids = _watch_ids(watch_name)
    subset = [r for r in results if r.get("id") in ids]
    n = len(subset)
    if n == 0:
        return 0.0, 0, 0
    correct = sum(1 for r in subset if r.get("correct"))
    return correct / n * 100, correct, n


slice_eval_records: dict[str, dict] = {}
_holdout_overall = _holdout_mcq = None

for slice_name in slice_results:
    data = slice_data[slice_name]
    results = slice_results[slice_name]

    mcq_res  = [r for r in results if r["is_mcq"]]
    free_res = [r for r in results if not r["is_mcq"]]
    multi_free = [
        r for r in free_res
        if n_ans_placeholders(next(d["question"] for d in data if d["id"] == r["id"])) > 1
    ]
    single_free = [r for r in free_res if r not in multi_free]

    mean_len = sum(len(r["response"]) for r in results) / len(results)
    mcq_emission = 0.0
    if mcq_res:
        emitted = 0
        for r in mcq_res:
            item = next(d for d in data if d["id"] == r["id"])
            n_opts = len(item.get("options") or [])
            if mcq_has_boxed_letter(r["response"], n_opts):
                emitted += 1
        mcq_emission = emitted / len(mcq_res) * 100

    len_delta_pct = (mean_len - BASELINE_MEAN_RESPONSE_LEN) / BASELINE_MEAN_RESPONSE_LEN * 100
    overall = acc(results)
    mcq_acc = acc(mcq_res)

    print("=" * 50)
    print(f"SFT EVAL — {SFT_RUN_NAME} @ step {EVAL_STEP} (slice={slice_name})")
    print("=" * 50)
    print(f"  LoRA       : {LORA_PATH or '(base model)'}")
    print(f"  PROMPT     : {PROMPT_VARIANT}")
    print(f"  MAX_TOKENS : {MAX_TOKENS}")
    print(f"  MCQ        : {sum(r['correct'] for r in mcq_res):4d} / {len(mcq_res):4d}  ({mcq_acc:.2f}%)")
    print(f"  Free-form  : {sum(r['correct'] for r in free_res):4d} / {len(free_res):4d}  ({acc(free_res):.2f}%)")
    if slice_name == "holdout":
        print(f"  Multi-blank: {sum(r['correct'] for r in multi_free):4d} / {len(multi_free):4d}  ({acc(multi_free):.2f}%)")
        print(f"  Single-blank: {sum(r['correct'] for r in single_free):4d} / {len(single_free):4d}  ({acc(single_free):.2f}%)")
    print(f"  Overall    : {sum(r['correct'] for r in results):4d} / {len(results):4d}  ({overall:.2f}%)")
    print("-" * 50)

    if slice_name == "holdout":
        _holdout_overall, _holdout_mcq = overall, mcq_acc
        geo_pct, geo_ok, geo_n = _watch_accuracy(results, "geometry")
        seq_pct, seq_ok, seq_n = _watch_accuracy(results, "sequences")
        q4_pct, q4_ok, q4_n = _watch_accuracy(results, "q4_long")
        mb_pct, mb_ok, mb_n = _watch_accuracy(results, "multi_blank_ge3")
        print(f"  Geometry             : {geo_ok:4d} / {geo_n:4d}  ({geo_pct:.2f}%)")
        print(f"  Sequences            : {seq_ok:4d} / {seq_n:4d}  ({seq_pct:.2f}%)")
        print(f"  Q4 long (≥435 chars) : {q4_ok:4d} / {q4_n:4d}  ({q4_pct:.2f}%)")
        print(f"  Multi-blank ≥3       : {mb_ok:4d} / {mb_n:4d}  ({mb_pct:.2f}%)")
        print("-" * 50)
    elif slice_name == "geometry":
        print(f"  (pub-002 ref ~{PUB002_GEOMETRY_ACC:.2f}%)")
        print("-" * 50)
        geo_pct = seq_pct = q4_pct = mb_pct = None
    elif slice_name == "prob_stats":
        print(f"  (pub-002 ref ~{PUB002_PROB_STATS_ACC:.2f}% full-public topic slice)")
        print("-" * 50)
        geo_pct = seq_pct = q4_pct = mb_pct = None
    else:
        geo_pct = seq_pct = q4_pct = mb_pct = None

    print(f"  MCQ \\boxed{{}} emission : {mcq_emission:.2f}%")
    print(f"  Mean response length  : {mean_len:.0f} chars ({len_delta_pct:+.1f}% vs baseline {BASELINE_MEAN_RESPONSE_LEN})")
    if len_delta_pct < -20:
        print("  WARNING: mean length dropped >20% — possible trace-style collapse")
    print("=" * 50)

    record = {
        "run_name": SFT_RUN_NAME,
        "eval_slice": slice_name,
        "eval_step": EVAL_STEP,
        "lora_path": LORA_PATH or None,
        "prompt_variant": PROMPT_VARIANT,
        "max_tokens": MAX_TOKENS,
        "n": len(results),
        "mcq_acc": mcq_acc,
        "free_acc": acc(free_res),
        "overall_acc": overall,
        "mcq_boxed_emission_pct": mcq_emission,
        "mean_response_len": mean_len,
        "mean_len_delta_pct": len_delta_pct,
    }
    if slice_name == "holdout":
        record.update({
            "multi_blank_acc": acc(multi_free),
            "single_blank_acc": acc(single_free),
            "geometry_acc": geo_pct,
            "sequences_acc": seq_pct,
            "q4_long_acc": q4_pct,
            "multi_blank_ge3_acc": mb_pct,
        })
    slice_eval_records[slice_name] = record

print("\n" + "=" * 50)
print("GATE (holdout)")
if _holdout_overall is None:
    print("  (no holdout run)")
else:
    ok_o = _holdout_overall >= GATE_HOLDOUT_OVERALL * 100
    ok_m = _holdout_mcq >= GATE_HOLDOUT_MCQ * 100
    print(f"  Overall {_holdout_overall:.2f}% vs {GATE_HOLDOUT_OVERALL*100:.2f}% → {'PASS' if ok_o else 'FAIL'}")
    print(f"  MCQ     {_holdout_mcq:.2f}% vs {GATE_HOLDOUT_MCQ*100:.2f}% → {'PASS' if ok_m else 'FAIL'}")
print("=" * 50)


SFT EVAL — openmath_sft007_5k @ step 0 (slice=holdout)
  LoRA       : /content/drive/MyDrive/CSE151B/checkpoints/openmath_sft007_5k/final_adapter
  PROMPT     : multi_blank
  MAX_TOKENS : 16384
  MCQ        :   59 /   75  (78.67%)
  Free-form  :   87 /  150  (58.00%)
  Multi-blank:   42 /   82  (51.22%)
  Single-blank:   45 /   68  (66.18%)
  Overall    :  146 /  225  (64.89%)
--------------------------------------------------
  Geometry             :   10 /   15  (66.67%)
  Sequences            :   10 /   13  (76.92%)
  Q4 long (≥435 chars) :   11 /   30  (36.67%)
  Multi-blank ≥3       :    7 /   20  (35.00%)
--------------------------------------------------
  MCQ \boxed{} emission : 98.67%
  Mean response length  : 9536 chars (-31.9% vs baseline 14000)
SFT EVAL — openmath_sft007_5k @ step 0 (slice=geometry)
  LoRA       : /content/drive/MyDrive/CSE151B/checkpoints/openmath_sft007_5k/final_adapter
  PROMPT     : multi_blank
  MAX_TOKENS : 16384
  MCQ        :   27 /   40  (67.50%)
 

## 10. Save results (per slice)

Same schema as the starter (`id`, `is_mcq`, `gold`, `response`, `correct`). Uses Drive when `DRIVE_PROJECT_ROOT` exists.


In [10]:
SAVE_EVAL = True

if not SAVE_EVAL:
    print("SAVE_EVAL=False — skipping write")
else:
    for slice_name, results in slice_results.items():
        eval_record = slice_eval_records[slice_name]
        eval_json_path = _eval_root / f"eval_{slice_name}_{EVAL_STEP}.json"
        out_path = _eval_root / f"eval_{slice_name}_{EVAL_STEP}.jsonl"

        with open(out_path, "w") as f:
            for r in results:
                record = {
                    "id": r["id"],
                    "is_mcq": r["is_mcq"],
                    "gold": r["gold"],
                    "response": r["response"],
                    "correct": r["correct"],
                }
                f.write(json.dumps(record) + "\n")

        with open(eval_json_path, "w") as f:
            json.dump(eval_record, f, indent=2)

        print(f"Saved {len(results)} records → {out_path.name}")
        print(f"Saved metrics → {eval_json_path.name}")


Saved 225 records → eval_holdout_0.jsonl
Saved metrics → eval_holdout_0.json
Saved 108 records → eval_geometry_0.jsonl
Saved metrics → eval_geometry_0.json
Saved 205 records → eval_prob_stats_0.jsonl
Saved metrics → eval_prob_stats_0.json


In [11]:
try:
    from google.colab import runtime
    runtime.unassign()
except ImportError:
    print("Not running in Colab — skipping runtime termination.")